In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

# Load the cleaned dataset from data folder
# Note: Adjust the file name if your saved cleaned CSV has a slightly different name
df = pd.read_csv(r'C:\Users\SSC\Desktop\job_intel\data\processed\jobs_clean.csv')

print(f"Dataset loaded successfully! Shape: {df.shape}")
print("\nColumns in dataset:")
print(df.columns.tolist())

Dataset loaded successfully! Shape: (85613, 17)

Columns in dataset:
['title', 'jobId', 'currency', 'jobUploaded', 'companyName', 'tagsAndSkills', 'experience', 'salary', 'location', 'companyId', 'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience']


In [15]:
# check non-null counts for salary and experience column

cols_to_check=['minimumSalary','maximumSalary','minimumExperience','maximumExperience']
print("Non-null counts (out of 85,613)")
print(df[cols_to_check].notnull().sum())

print("\nCurrency Value Counts:")
print(df['currency'].value_counts(dropna=False))

print("\nSalary & Experience Summary Statistics:")

print(df[['minimumSalary','maximumSalary','minimumExperience','maximumExperience']].describe())

Non-null counts (out of 85,613)
minimumSalary        30024
maximumSalary        30032
minimumExperience    85613
maximumExperience    85613
dtype: int64

Currency Value Counts:
currency
INR    85613
Name: count, dtype: int64

Salary & Experience Summary Statistics:
       minimumSalary  maximumSalary  minimumExperience  maximumExperience
count   3.002400e+04   3.003200e+04       85613.000000       85613.000000
mean    5.841127e+05   9.361421e+05           4.003714           7.812225
std     8.647058e+05   1.255082e+06           3.568273           4.490035
min     1.000000e+03   1.000000e+03           0.000000           0.000000
25%     2.000000e+05   3.500000e+05           1.000000           5.000000
50%     3.000000e+05   5.500000e+05           3.000000           7.000000
75%     6.000000e+05   1.000000e+06           5.000000          10.000000
max     7.000000e+07   9.500000e+07          30.000000          35.000000


In [16]:
# 1. Keep only rows where both minimum and maximum salary are present
df_sal = df.dropna(subset=['minimumSalary', 'maximumSalary']).copy()

# 2. Calculate average salary in LPA (Lakhs Per Annum) and average experience
df_sal['avg_salary_lpa'] = ((df_sal['minimumSalary'] + df_sal['maximumSalary']) / 2) / 100000
df_sal['avg_experience'] = (df_sal['minimumExperience'] + df_sal['maximumExperience']) / 2

# 3. Filter out extreme outliers/typos (keep realistic range: 1 LPA to 100 LPA)
df_sal = df_sal[(df_sal['avg_salary_lpa'] >= 1.0) & (df_sal['avg_salary_lpa'] <= 100.0)].copy()

print(f"Clean Salary Dataset Shape: {df_sal.shape}")
print("\nTarget Variable (avg_salary_lpa) Summary:")
print(df_sal[['avg_salary_lpa', 'avg_experience']].describe())

Clean Salary Dataset Shape: (29602, 19)

Target Variable (avg_salary_lpa) Summary:
       avg_salary_lpa  avg_experience
count    29602.000000    29602.000000
mean         7.593890        4.794676
std          8.783283        3.747465
min          1.000000        0.000000
25%          3.000000        2.500000
50%          4.250000        3.500000
75%          8.500000        6.500000
max         97.500000       30.500000


In [17]:
# Check exact salary values at 90%, 95%, 98%, and 99%
percentiles = df_sal['avg_salary_lpa'].quantile([0.90, 0.95, 0.98, 0.99])
print("Salary Percentiles (in LPA):")
print(percentiles)

Salary Percentiles (in LPA):
0.90    17.5
0.95    25.0
0.98    35.0
0.99    45.0
Name: avg_salary_lpa, dtype: float64


In [18]:
# 1. Keep all valid salary rows without strict high-end trimming
df_sal = df.dropna(subset=['minimumSalary', 'maximumSalary']).copy()

# 2. Calculate average salary in LPA and average experience
df_sal['avg_salary_lpa'] = ((df_sal['minimumSalary'] + df_sal['maximumSalary']) / 2) / 100000
df_sal['avg_experience'] = (df_sal['minimumExperience'] + df_sal['maximumExperience']) / 2

# 3. Filter only basic typo outliers (e.g., salary less than 1 LPA or greater than 100 LPA)
df_sal = df_sal[(df_sal['avg_salary_lpa'] >= 1.0) & (df_sal['avg_salary_lpa'] <= 100.0)].copy()

print(f"Total Salary Rows Kept (with high-end values intact): {df_sal.shape[0]}")
print(df_sal[['avg_salary_lpa', 'avg_experience']].describe())

Total Salary Rows Kept (with high-end values intact): 29602
       avg_salary_lpa  avg_experience
count    29602.000000    29602.000000
mean         7.593890        4.794676
std          8.783283        3.747465
min          1.000000        0.000000
25%          3.000000        2.500000
50%          4.250000        3.500000
75%          8.500000        6.500000
max         97.500000       30.500000


In [19]:
from sklearn.model_selection import train_test_split

# apply log transformation on target salary
df_sal['log_salary'] = np.log1p(df_sal['avg_salary_lpa'])

# prepare text features and clean meaning values in location

df_sal['text_features'] = df_sal['jobDescription'].fillna('') + ' ' + df_sal['tagsAndSkills'].fillna('')
df_sal['location'] = df_sal['location'].fillna('Unknown')

# define inpur features (X) and Target (y)

X = df_sal[['avg_experience', 'location','text_features']]
y = df_sal['log_salary']

# split data into 80% training and 20 testing sets

X_train , X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")




X_train shape: (23681, 3)
X_test shape: (5921, 3)


In [20]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingRegressor

# 1. Preprocessor for Numerical, Categorical, and Text columns
preprocessor = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', ['avg_experience']),
        ('cat', OneHotEncoder(handle_unknown='ignore', max_categories=30, sparse_output=False), ['location']),
        ('txt', TfidfVectorizer(max_features=500, stop_words='english'), 'text_features')
    ],
    sparse_threshold=0.0  # Converts final combined output to dense array for HistGradientBoosting
)

# 2. Complete pipeline: Preprocessor + Model
salary_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', HistGradientBoostingRegressor(max_iter=200, learning_rate=0.1, random_state=42))
])

# 3. Train the pipeline on training data
print("Training Salary Predictor... (this may take 15-30 seconds)")
salary_pipeline.fit(X_train, y_train)
print("Training Complete!")

Training Salary Predictor... (this may take 15-30 seconds)
Training Complete!


In [21]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Predict on test data (predictions will be in log scale)
y_pred_log = salary_pipeline.predict(X_test)

# 2. Convert both predictions and actual test values back to original LPA scale
y_pred_lpa = np.expm1(y_pred_log)
y_test_lpa = np.expm1(y_test)

# 3. Calculate Evaluation Metrics on both Log Scale and Actual LPA Scale
r2_log = r2_score(y_test, y_pred_log)
r2_lpa = r2_score(y_test_lpa, y_pred_lpa)
mae_lpa = mean_absolute_error(y_test_lpa, y_pred_lpa)
rmse_lpa = np.sqrt(mean_squared_error(y_test_lpa, y_pred_lpa))

# 4. Print the results clearly
print("--- Salary Predictor Evaluation Results ---")
print(f"R2 Score (Log Scale):    {r2_log:.4f}")
print(f"R2 Score (Actual LPA):   {r2_lpa:.4f}")
print(f"MAE (Mean Abs Error):    {mae_lpa:.2f} LPA")
print(f"RMSE (Root Mean Sq Err): {rmse_lpa:.2f} LPA")

--- Salary Predictor Evaluation Results ---
R2 Score (Log Scale):    0.6920
R2 Score (Actual LPA):   0.5372
MAE (Mean Abs Error):    2.70 LPA
RMSE (Root Mean Sq Err): 6.09 LPA


In [22]:
import numpy as np
import pandas as pd

y_train_lpa = np.expm1(y_train)

# Simple tareeka: sabko ek hi salary (train ki median) bol do
simple_pred = np.full(len(y_test_lpa), y_train_lpa.median())

print("Simple tareeka MAE :", round(mean_absolute_error(y_test_lpa, simple_pred), 2), "LPA")
print("Mera model MAE     :", round(mean_absolute_error(y_test_lpa, y_pred_lpa), 2), "LPA")

Simple tareeka MAE : 4.65 LPA
Mera model MAE     : 2.7 LPA


In [23]:
print(X_train.columns.tolist())
print(X_train["text_features"].str.contains(r"lpa|lacs|lakh|ctc|per annum", case=False).mean())

['avg_experience', 'location', 'text_features']
0.06296186816435116


In [26]:
# 1. Add 'title' along with 'jobDescription' and 'tagsAndSkills' into text_features
df_sal['text_features'] = (
    df_sal['title'].fillna('') + ' ' + 
    df_sal['jobDescription'].fillna('') + ' ' + 
    df_sal['tagsAndSkills'].fillna('')
)

# 2. Define X and y with updated text_features
X = df_sal[['avg_experience', 'location', 'text_features']]
y = df_sal['log_salary']

# 3. Re-split into 80% Train and 20% Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. Re-train the existing pipeline on updated features
print("Re-training Salary Predictor with Job Titles included...")
salary_pipeline.fit(X_train, y_train)

# 5. Predict and evaluate
y_pred_log = salary_pipeline.predict(X_test)
y_pred_lpa = np.expm1(y_pred_log)
y_test_lpa = np.expm1(y_test)

print("\n--- Updated Evaluation Results (With Title) ---")
print(f"R2 Score (Log Scale):    {r2_score(y_test, y_pred_log):.4f}  (Before: 0.6920)")
print(f"R2 Score (Actual LPA):   {r2_score(y_test_lpa, y_pred_lpa):.4f}  (Before: 0.5372)")
print(f"MAE (Mean Abs Error):    {mean_absolute_error(y_test_lpa, y_pred_lpa):.2f} LPA (Before: 2.70 LPA)")
print(f"RMSE (Root Mean Sq Err): {np.sqrt(mean_squared_error(y_test_lpa, y_pred_lpa)):.2f} LPA (Before: 6.09 LPA)")

Re-training Salary Predictor with Job Titles included...

--- Updated Evaluation Results (With Title) ---
R2 Score (Log Scale):    0.7048  (Before: 0.6920)
R2 Score (Actual LPA):   0.5396  (Before: 0.5372)
MAE (Mean Abs Error):    2.64 LPA (Before: 2.70 LPA)
RMSE (Root Mean Sq Err): 6.08 LPA (Before: 6.09 LPA)


In [25]:
print(df_sal.columns.tolist())

['title', 'jobId', 'currency', 'jobUploaded', 'companyName', 'tagsAndSkills', 'experience', 'salary', 'location', 'companyId', 'ReviewsCount', 'AggregateRating', 'jobDescription', 'minimumSalary', 'maximumSalary', 'minimumExperience', 'maximumExperience', 'avg_salary_lpa', 'avg_experience', 'log_salary', 'text_features']


In [27]:
from sklearn.linear_model import Ridge

# 1. Create a pipeline with the exact same preprocessor, but using Ridge Regression
ridge_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0, random_state=42))
])

# 2. Train the Ridge model
print("Training Ridge Regression Baseline...")
ridge_pipeline.fit(X_train, y_train)

# 3. Predict on X_test and convert from log scale back to LPA
ridge_pred_log = ridge_pipeline.predict(X_test)
ridge_pred_lpa = np.expm1(ridge_pred_log)

# 4. Create a clean side-by-side Model Comparison Table
model_comparison = pd.DataFrame({
    'Model': ['Ridge Regression (Baseline)', 'HistGradientBoosting (With Title)'],
    'R2 Score (Log)': [
        round(r2_score(y_test, ridge_pred_log), 4),
        round(r2_score(y_test, y_pred_log), 4)
    ],
    'R2 Score (Actual LPA)': [
        round(r2_score(y_test_lpa, ridge_pred_lpa), 4),
        round(r2_score(y_test_lpa, y_pred_lpa), 4)
    ],
    'MAE (LPA)': [
        round(mean_absolute_error(y_test_lpa, ridge_pred_lpa), 2),
        round(mean_absolute_error(y_test_lpa, y_pred_lpa), 2)
    ],
    'RMSE (LPA)': [
        round(np.sqrt(mean_squared_error(y_test_lpa, ridge_pred_lpa)), 2),
        round(np.sqrt(mean_squared_error(y_test_lpa, y_pred_lpa)), 2)
    ]
})

print("\n--- Final Model Comparison Table ---")
display(model_comparison)

Training Ridge Regression Baseline...

--- Final Model Comparison Table ---


,Model,R2 Score (Log),R2 Score (Actual LPA),MAE (LPA),RMSE (LPA)
0,Ridge Regression (Baseline),0.6748,0.4803,2.85,6.45
1,HistGradientBoosting (With Title),0.7048,0.5396,2.64,6.08


In [28]:
# 1. Create a DataFrame with Actual Salary, Predicted Salary (from HGBR), and Absolute Error
error_df = pd.DataFrame({
    'Actual_LPA': y_test_lpa,
    'Pred_LPA': y_pred_lpa
})
error_df['Abs_Error'] = (error_df['Actual_LPA'] - error_df['Pred_LPA']).abs()

# 2. Divide Actual Salary into 5 meaningful buckets using pd.cut()
bins = [0, 5, 10, 20, 40, 100]
labels = ['0-5 LPA', '5-10 LPA', '10-20 LPA', '20-40 LPA', '40+ LPA']
error_df['Salary_Bucket'] = pd.cut(error_df['Actual_LPA'], bins=bins, labels=labels)

# 3. Group by each bucket and calculate Count, Average Actual, Average Predicted, and MAE
bucket_summary = error_df.groupby('Salary_Bucket', observed=False).agg(
    Job_Count=('Actual_LPA', 'count'),
    Avg_Actual_LPA=('Actual_LPA', 'mean'),
    Avg_Pred_LPA=('Pred_LPA', 'mean'),
    Bucket_MAE=('Abs_Error', 'mean')
).round(2)

print("--- Bucket-wise Error Analysis (HistGradientBoosting) ---")
display(bucket_summary)

--- Bucket-wise Error Analysis (HistGradientBoosting) ---


,Job_Count,Avg_Actual_LPA,Avg_Pred_LPA,Bucket_MAE
Salary_Bucket,,,,
0-5 LPA,3602,3.16,3.76,0.99
5-10 LPA,1094,6.83,7.01,2.12
10-20 LPA,774,14.08,12.85,4.38
20-40 LPA,368,27.25,18.85,9.44
40+ LPA,83,55.83,21.13,34.70


In [30]:
import os
import joblib

# 1. Ensure the 'models' directory exists
os.makedirs('models', exist_ok=True)

# 2. Define the save path
model_path = 'models/salary_pipeline.pkl'

# 3. Save the trained salary pipeline
joblib.dump(salary_pipeline, model_path)

print(f"Success! Salary Predictor pipeline saved successfully at: {model_path}")

Success! Salary Predictor pipeline saved successfully at: models/salary_pipeline.pkl
